# KKBox Churn Intelligence — Exploratory Data Analysis

Key questions:
1. What does churn rate look like — is it imbalanced?
2. Which transaction behaviours correlate with churn?
3. Does listening activity in the final 30 days predict churn?
4. What features correlate most strongly with churn?

In [ ]:
import os, sys
os.chdir(r'.')
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
print('Libraries loaded.')

In [ ]:
df = pd.read_parquet('data/processed/master.parquet')
print(f'Shape: {df.shape}')
print(f'Churn rate: {df.is_churn.mean():.3%}')
df.head(3)

## 1. Class Imbalance

A naive model predicting everyone as non-churn achieves 91% accuracy — which is why we use AUC-PR, not accuracy.

In [ ]:
churn_counts = df['is_churn'].value_counts()
labels = ['No Churn', 'Churn']
colors = ['#2563eb', '#dc2626']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(labels, churn_counts.values, color=colors, width=0.4)
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 5000, f'{v:,}', ha='center', fontweight='bold')
axes[0].set_title('Churn vs No Churn')
axes[0].set_ylabel('Users')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
axes[1].pie(churn_counts.values, labels=labels, colors=colors, autopct='%1.2f%%', startangle=90)
axes[1].set_title('Churn Rate Distribution')
plt.tight_layout()
plt.show()
print(f'Imbalance ratio: {churn_counts[0]/churn_counts[1]:.1f}:1')

## 2. Transaction Behaviour vs Churn

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

auto_renew_churn = df.groupby('last_auto_renew')['is_churn'].mean()
axes[0].bar(['Auto-Renew OFF', 'Auto-Renew ON'], [auto_renew_churn.get(0,0), auto_renew_churn.get(1,0)], color=['#dc2626','#16a34a'])
axes[0].set_title('Churn Rate by Auto-Renew')
axes[0].set_ylabel('Churn Rate')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1%}'))
for i, v in enumerate([auto_renew_churn.get(0,0), auto_renew_churn.get(1,0)]):
    axes[0].text(i, v+0.005, f'{v:.1%}', ha='center', fontweight='bold')

cancel_churn = df.groupby('last_is_cancel')['is_churn'].mean()
axes[1].bar(['No Cancel','Cancelled'], [cancel_churn.get(0,0), cancel_churn.get(1,0)], color=['#16a34a','#dc2626'])
axes[1].set_title('Churn Rate by Last Cancel')
axes[1].set_ylabel('Churn Rate')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1%}'))
for i, v in enumerate([cancel_churn.get(0,0), cancel_churn.get(1,0)]):
    axes[1].text(i, v+0.005, f'{v:.1%}', ha='center', fontweight='bold')

plan_bins = pd.cut(df['last_plan_days'].fillna(30), bins=[0,30,90,180,365,2000], labels=['Monthly','90d','180d','Annual','2yr+'])
plan_churn = df.groupby(plan_bins)['is_churn'].mean()
axes[2].bar(plan_churn.index, plan_churn.values, color='#7c3aed')
axes[2].set_title('Churn Rate by Plan Duration')
axes[2].set_ylabel('Churn Rate')
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1%}'))

plt.tight_layout()
plt.show()

## 3. Listening Activity vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

churned = df[df['is_churn']==1]['active_days_30d'].fillna(0)
not_churned = df[df['is_churn']==0]['active_days_30d'].fillna(0)
axes[0].hist(not_churned, bins=31, alpha=0.6, color='#2563eb', label='No Churn', density=True)
axes[0].hist(churned, bins=31, alpha=0.6, color='#dc2626', label='Churn', density=True)
axes[0].set_xlabel('Active Days in Last 30 Days')
axes[0].set_ylabel('Density')
axes[0].set_title('Listening Activity (30d) by Churn')
axes[0].legend()

silent_churn = df[df['active_days_30d'].fillna(0)==0]['is_churn'].mean()
non_silent_churn = df[df['active_days_30d'].fillna(0)>0]['is_churn'].mean()
axes[1].bar(['Active in March','Silent in March'], [non_silent_churn, silent_churn], color=['#16a34a','#dc2626'])
axes[1].set_title('Churn Rate: Silent vs Active')
axes[1].set_ylabel('Churn Rate')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1%}'))
for i, v in enumerate([non_silent_churn, silent_churn]):
    axes[1].text(i, v+0.005, f'{v:.1%}', ha='center', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.show()
print(f'Silent churn rate: {silent_churn:.2%}')
print(f'Active churn rate: {non_silent_churn:.2%}')
print(f'Lift: {silent_churn/non_silent_churn:.1f}x')

## 4. Days to Expiry vs Churn

55% of LightGBM feature importance. NOT leakage — comes from transaction table, not the churn label.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

dte = df['days_to_expiry'].clip(-30, 180).fillna(30)
axes[0].hist(dte[df['is_churn']==0], bins=50, alpha=0.6, color='#2563eb', label='No Churn', density=True)
axes[0].hist(dte[df['is_churn']==1], bins=50, alpha=0.6, color='#dc2626', label='Churn', density=True)
axes[0].axvline(0, color='black', linestyle='--', lw=1.5, label='Expiry date')
axes[0].set_xlabel('Days to Expiry')
axes[0].set_ylabel('Density')
axes[0].set_title('Days to Expiry by Churn')
axes[0].legend()

expiry_bins = pd.cut(dte, bins=[-30,0,7,30,90,180], labels=['Expired','0-7d','7-30d','30-90d','90-180d'])
expiry_churn = df.groupby(expiry_bins)['is_churn'].mean()
axes[1].bar(expiry_churn.index, expiry_churn.values, color='#f59e0b')
axes[1].set_title('Churn Rate by Expiry Bucket')
axes[1].set_ylabel('Churn Rate')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1%}'))

plt.tight_layout()
plt.show()

## 5. Feature Correlation with Churn

In [ ]:
from src.feature_engineering import build_features
X, y, features = build_features(df)

correlations = X.corrwith(y.astype(float)).abs().sort_values(ascending=False)
top20 = correlations.head(20)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top20.index[::-1], top20.values[::-1], color='#2563eb')
ax.set_xlabel('|Correlation with Churn|')
ax.set_title('Top 20 Features by Correlation with Churn')
ax.bar_label(bars, fmt='{:.3f}', padding=3, fontsize=9)
plt.tight_layout()
plt.show()

print('Top 10:')
print(correlations.head(10).to_string())

## 6. Key Findings

**Finding 1 — 91:9 class imbalance** — accuracy is useless. AUC-PR is the correct metric.

**Finding 2 — Auto-renew OFF is the strongest behavioural signal** — validates `auto_renew_ratio` and `last_auto_renew`.

**Finding 3 — Silent users churn significantly more** — `silent_march` binary flag captures this directly.

**Finding 4 — Days to expiry dominates at 55% feature importance** — legitimate domain signal, not leakage.

**Finding 5 — Monthly plan users churn more than annual** — validates `short_plan_risk` feature.

**Model choice: LightGBM with 5-fold StratifiedKFold** — native imbalance handling, fast on 970k rows, SHAP native support.